# LLM Model Benchmarking Pipeline

In [31]:
MODEL_NAME = "gemma3:270m"

## Ollama Setup

In [44]:
%%bash
nvidia-smi

Tue Nov  4 20:38:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   39C    P8              1W /  100W |     560MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [45]:
# %%bash
# curl -fsSL https://ollama.com/install.sh | sh

In [46]:
import requests
import time
import subprocess

In [47]:
# Start Ollama server in background
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait until server responds
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=1)
        if r.ok:
            print("[ollama server] Ollama server ready")
            break
    except Exception as _:
        print(f"[ollama server] Waiting for server... ({i+1}/30)")
        time.sleep(1)

[ollama server] Waiting for server... (1/30)
[ollama server] Ollama server ready


In [48]:
# List of models you want to download
print(f"[model download] Downloading {MODEL_NAME} ...")
try:
    # Run the Ollama pull command
    result = subprocess.run(
        ["ollama", "pull", MODEL_NAME],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=True
    )
    print(f"[model download] Successfully downloaded {MODEL_NAME}\n")
except subprocess.CalledProcessError as e:
    print(f"[model download] Failed to download {MODEL_NAME}")
    print(e.stderr)


[model download] Downloading gemma3:270m ...
[model download] Successfully downloaded gemma3:270m



In [49]:
# Test URL
url = "http://localhost:11434/api/tags"

print("Checking if Ollama server is running...")

for i in range(30):  # wait up to ~30 seconds
    try:
        r = requests.get(url, timeout=2)
        if r.ok:
            print("[ollama server] Ollama server is running!")
            print("Available models:")
            k = r.json()["models"]
            for a, m in enumerate(k):
                print(a, m["name"])
            break
    except Exception as _:
        print(f"[ollama server] Waiting for server... ({i+1}/30)")
        time.sleep(1)
else:
    print("[ollama server] Ollama server not responding.")


Checking if Ollama server is running...
[ollama server] Ollama server is running!
Available models:
0 gemma3:270m
1 mistral:7b
2 gemma3:12b
3 llama3.1:8b
4 qwen3:8b


In [50]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model=MODEL_NAME, messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

The sky is blue because of a phenomenon called **Rayleigh Scattering**. Here's a breakdown:

*   **Sunlight:** Sunlight is a form of electromagnetic radiation that travels through the Earth's atmosphere.
*   **Atmospheric Particles:** The atmosphere contains a variety of particles, including dust, gases, and aerosols.
*   **Rayleigh Scattering:** These particles scatter light in different directions. When sunlight passes through the atmosphere, the particles interact with the light and scatter it.
*   **Blue Light:** This scattering of light is what we see as blue.
*   **Why Blue?** Because the blue light is scattered more by the particles in the atmosphere than the red light that is reflected. This means that the sky appears blue.


The sky is blue because of a phenomenon called **Rayleigh Scattering**. Here's a breakdown:

*   **Sunlight:** Sunlight is a form of electromagnetic radiation that travels through the Earth's atmosphere.
*   **Atmospheric Particles:** The atmosphere contai

## Importing Packages for Execution

In [59]:
import os
import json
import time

from ollama import generate

## Utility Functions

In [53]:
def json_to_dict(file_path: str) -> dict:
    data_dict = {}
    with open(file_path, 'r') as data:
        data_dict = json.load(data)

    return data_dict

In [14]:
def csv_to_dict(file_path: str) -> dict:
    data_list = []
    data_dict = {}

    with open(file_path, 'r') as data:
        data_list = data.readlines()

    for elements in data_list[1:]:
        id, sys, prompt = elements.split(',')
        data_dict[id] = {
            "system_prompt": sys.strip(),
            "test_prompt": prompt.strip()
        }

    return data_dict

In [40]:
def inference_llm(sys_prompt: str, text_prompt: str, model_name=MODEL_NAME) -> str:
    output = generate(model=model_name, prompt=text_prompt, system=sys_prompt)
    return output.response
    

## Training
- Goal: read different prompts from csv file and do inferences, save the output to another csv file with unique_id.
- This is a skeleton file, a singular function should trigger the training process and save the output to csv at every k prompts.

### Steps:
1. Load csv data into dictionary {unique_id: {system_prompt: "prompt", test_prompt: "prompt"}}
2. Loop through data and call llm_inference function to get output from llm
3. Store output into dictionary {unique_id: output}
4. Save the dictionary at every k prompts in a csv

In [ ]:
folder_path = "/workspaces/Project/CSDS555-ResAI-Final-Project-Research/data/"
input_path = os.path.join(folder_path, "input")
output_path = os.path.join(folder_path, "output")

In [57]:
# Step 1 CSV input
# input_file = os.path.join(input_path, "test_data.csv")
# input_dict = csv_to_dict(input_file)

# Step 1 JSON input
input_file = os.path.join(input_path, "test_data.json")
input_dict = json_to_dict(input_file)

print(input_dict)

{'01': {'system_prompt': 'Always answer max in 2 lines.', 'test_prompt': 'You are an expert mathematician,solve this calculus problem: find the derivative of x^3 + 4x^2 - 7x + 10'}, '02': {'system_prompt': 'Always answer max in 2 lines.', 'test_prompt': "You are an expert physicist,explain how Newton's third law applies to rocket propulsion"}, '03': {'system_prompt': 'Always answer max in 2 lines.', 'test_prompt': 'You are an expert chemist,describe the process of balancing a redox reaction'}, '04': {'system_prompt': 'Always answer max in 2 lines.', 'test_prompt': 'You are an expert biologist,explain how photosynthesis converts light energy into chemical energy'}, '05': {'system_prompt': 'Always answer max in 2 lines.', 'test_prompt': 'You are an expert historian,write a short summary about the causes of World War I'}, '06': {'system_prompt': 'Always answer max in 2 lines.', 'test_prompt': 'You are an expert linguist,analyze the syntax and structure of the sentence "She quickly ran hom

In [60]:
# Step 2 JSON output
output_file = os.path.join(output_path, f"test_data_{time.time_ns()}.json")  # Create new file if already exists
output_data = {}
for i, (id, prompt) in enumerate(input_dict.items()):
    if (i % 5) == 0:
        with open(output_file, 'w') as op_file:
            json.dump(output_data, op_file, indent=2)
    
    output = inference_llm(sys_prompt=prompt["system_prompt"], text_prompt=prompt["test_prompt"])
    output_data[id] = output

# Final load if any are missed
with open(output_file, 'w') as op_file:
            json.dump(output_data, op_file, indent=2)